# 第6章　実践：CNN で手書き数字を分類（MNIST）

画像認識の定番 **CNN（畳み込みニューラルネット）** を作り、手書き数字 0〜9 を分類します。
ここまでの全部（テンソル・自動微分・学習ループ・nn.Module・DataLoader）を1本に統合します。

この章のゴール：CNN を組んで MNIST で 98% 程度の正解率を出し、GPU の使い方を知る。

> **このノートの使い方**
> - 上から順にセルを実行（Colab/Jupyter ともに `Shift + Enter`）。
> - コードは**少し書き換えて壊して直す**のが一番伸びます。各章末に演習があります。
> - GPU は不要な章が多いです。重い章（CNN）では使い方を案内します。

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA (GPU) available:", torch.cuda.is_available())

## 6-0. GPU を有効にする（Colab）
重い章なので GPU 推奨。**メニュー「ランタイム」→「ランタイムのタイプを変更」→ GPU (T4)** を選び、最初のセルを再実行。
GPU が無くても CPU で動きます（少し遅いだけ。epoch を減らせばOK）。

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 6-1. 画像テンソルの形と「畳み込み」の直感

- 画像バッチの形は **`(N, C, H, W)`** = (枚数, チャンネル, 高さ, 幅)。MNIST は白黒なので `C=1`。
- **畳み込み (`nn.Conv2d`)**：小さなフィルタを画像上で滑らせ、「エッジ」「曲線」などの**局所パターン**を検出。
- **プーリング (`nn.MaxPool2d`)**：画像を縮小して情報を要約（位置ずれに強くなる・計算減）。

まず Conv が形をどう変えるか確認します。

In [ ]:
conv = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1)
dummy = torch.randn(4, 1, 28, 28)        # 4枚の28x28白黒
out = conv(dummy)
print("入力:", dummy.shape, "-> 畳み込み後:", out.shape)   # (4, 8, 28, 28)
print("プーリング後:", nn.MaxPool2d(2)(out).shape)         # (4, 8, 14, 14)

## 6-2. データを用意

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
train_ds = datasets.MNIST("./data", train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST("./data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=64,  shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=1000, shuffle=False)
print("train:", len(train_ds), " test:", len(test_ds))

## 6-3. CNN モデルを定義
`Conv → ReLU → Pool` を2回 → 平らに伸ばして(`Flatten`)全結合層で10クラスへ。

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 28->14
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 14->7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),              # (N,32,7,7) -> (N, 32*7*7)
            nn.Linear(32 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, 10),        # 10クラス（数字0〜9）
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = CNN().to(device)      # モデルを GPU/CPU へ
print(model)

## 6-4. 学習する
ポイント：**バッチごとに `xb, yb = xb.to(device), yb.to(device)`** でデータも同じデバイスへ送る。

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 3   # CPU なら 1 でもOK。GPU なら 3〜5 で 98%+ になる
model.train()
for epoch in range(EPOCHS):
    running = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)   # データもデバイスへ
        optimizer.zero_grad()        # ①
        logits = model(xb)           # ②
        loss = criterion(logits, yb) # ③
        loss.backward()              # ④
        optimizer.step()             # ⑤
        running += loss.item()
    print(f"epoch {epoch+1}: 平均loss = {running/len(train_loader):.4f}")

## 6-5. テストデータで正解率を測る

In [ ]:
model.eval()
correct = total = 0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb).argmax(dim=1)
        correct += (pred == yb).sum().item()
        total += yb.size(0)
print(f"テスト正解率: {100*correct/total:.2f}%")

## 6-6. 予測を可視化（どこで間違えた？）

In [ ]:
import matplotlib.pyplot as plt
xb, yb = next(iter(test_loader))
with torch.no_grad():
    pred = model(xb.to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 6, figsize=(11, 4))
for ax, i in zip(axes.flat, range(12)):
    ax.imshow(xb[i].squeeze(), cmap="gray")
    ok = pred[i].item() == yb[i].item()
    ax.set_title(f"pred {pred[i].item()} / true {yb[i].item()}",
                 color=("green" if ok else "red"), fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 演習 6
1. `EPOCHS` を増やす／`lr` を変えると正解率はどうなる？
2. Conv 層をもう1段増やす、`out_channels` を変えるなど構造をいじってみよう（`Flatten` 後の入力サイズに注意：形が変わったら `nn.Linear` の数も合わせる）。
3. データを **CIFAR-10**（カラー画像、`C=3`）に変えてみよう（`datasets.CIFAR10`、`Conv2d` の `in_channels=3`、画像32x32に合わせてサイズ計算）。難易度が上がるのを体感できる。

In [ ]:
# ここに自分のコードを書いて実行してみよう
